In [2]:
import pandas
import os

# This query represents dataset "Women with BC within 5 years after inclusion"

dataset_breast_cancer_5y_person_sql = """
WITH enrollment AS (
  -- Date administrative d'inclusion = 1ère date du module "Consent PII"
  SELECT
    o.person_id,
    MIN(o.observation_date) AS inclusion_date
  FROM `""" + os.environ["WORKSPACE_CDR"] + """.concept` c
  JOIN `""" + os.environ["WORKSPACE_CDR"] + """.concept_ancestor` ca
    ON ca.ancestor_concept_id = c.concept_id
  JOIN `""" + os.environ["WORKSPACE_CDR"] + """.observation` o
    ON o.observation_concept_id = ca.descendant_concept_id
  WHERE c.concept_name = 'Consent PII'
    AND c.concept_class_id = 'Module'
  GROUP BY o.person_id
),
breast_cancer_concepts AS (
  -- Cancer du sein (4157332 + descendants via Dataset Builder)
  SELECT DISTINCT c.concept_id
  FROM `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c
  JOIN (
    SELECT CAST(cr.id AS STRING) AS id
    FROM `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr
    WHERE cr.concept_id IN (4157332)
      AND cr.full_text LIKE '%_rank1]%'
  ) a
    ON (c.path LIKE CONCAT('%.', a.id, '.%')
        OR c.path LIKE CONCAT('%.', a.id)
        OR c.path LIKE CONCAT(a.id, '.%')
        OR c.path = a.id)
  WHERE c.is_standard = 1
    AND c.is_selectable = 1
),
breast_cancer_5y AS (
  -- 1er diagnostic de cancer du sein ≤ 5 ans après inclusion
  SELECT
    co.person_id,
    MIN(co.condition_start_date) AS first_breast_cancer_date
  FROM `""" + os.environ["WORKSPACE_CDR"] + """.condition_occurrence` co
  JOIN enrollment en
    ON en.person_id = co.person_id
  JOIN breast_cancer_concepts bc
    ON bc.concept_id = co.condition_concept_id
  WHERE co.condition_start_date IS NOT NULL
    AND co.condition_start_date > en.inclusion_date
    AND co.condition_start_date <= DATE_ADD(en.inclusion_date, INTERVAL 5 YEAR)
  GROUP BY co.person_id
)

SELECT
  person.person_id,
  p_gender_concept.concept_name AS gender,
  person.birth_datetime AS date_of_birth,
  p_ethnicity_concept.concept_name AS ethnicity,
  p_self_reported_category_concept.concept_name AS self_reported_category,
  en.inclusion_date,
  bc5.first_breast_cancer_date
FROM `""" + os.environ["WORKSPACE_CDR"] + """.person` person
JOIN breast_cancer_5y bc5
  ON bc5.person_id = person.person_id
JOIN enrollment en
  ON en.person_id = person.person_id
LEFT JOIN `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept
  ON person.gender_concept_id = p_gender_concept.concept_id
LEFT JOIN `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity_concept
  ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id
LEFT JOIN `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_self_reported_category_concept
  ON person.self_reported_category_concept_id = p_self_reported_category_concept.concept_id
WHERE person.gender_concept_id = 45878463  -- FEMMES UNIQUEMENT
"""

dataset_breast_cancer_5y_person_df = pandas.read_gbq(
    dataset_breast_cancer_5y_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook"
)

dataset_breast_cancer_5y_person_df


Downloading:   0%|          | 0/7920 [00:00<?, ?rows/s]

,person_id,gender,date_of_birth,ethnicity,self_reported_category,inclusion_date,first_breast_cancer_date
0,3088025,Female,1935-06-15 00:00:00+00:00,What Race Ethnicity: Race Ethnicity None Of These,None of these,2019-06-12,2019-07-11
1,1010524,Female,1945-06-15 00:00:00+00:00,What Race Ethnicity: Race Ethnicity None Of These,None of these,2018-07-27,2019-03-01
2,4591164,Female,1959-06-15 00:00:00+00:00,What Race Ethnicity: Race Ethnicity None Of These,None of these,2021-04-28,2021-05-28
3,1230270,Female,1947-06-15 00:00:00+00:00,What Race Ethnicity: Race Ethnicity None Of These,None of these,2018-06-11,2018-12-05
4,8160515,Female,1976-06-15 00:00:00+00:00,What Race Ethnicity: Race Ethnicity None Of These,None of these,2021-05-12,2021-05-17
...,...,...,...,...,...,...,...
7915,7547182,Female,1970-06-15 00:00:00+00:00,Hispanic or Latino,What Race Ethnicity: Hispanic,2023-03-09,2023-04-10
7916,2232111,Female,1975-06-15 00:00:00+00:00,Hispanic or Latino,What Race Ethnicity: Hispanic,2019-06-26,2022-08-31
7917,2767168,Female,1977-06-15 00:00:00+00:00,Hispanic or Latino,What Race Ethnicity: Hispanic,2019-03-05,2023-03-31
7918,2576857,Female,1978-06-15 00:00:00+00:00,Hispanic or Latino,What Race Ethnicity: Hispanic,2018-01-17,2018-01-22


In [3]:
import pandas
import os

# This query represents dataset "Women with BC within 5 years after inclusion"

dataset_no_prevalent_breast_cancer_women_sql = """
WITH enrollment AS (
  -- Date administrative d'inclusion = 1ère date du module "Consent PII"
  SELECT
    o.person_id,
    MIN(o.observation_date) AS inclusion_date
  FROM `""" + os.environ["WORKSPACE_CDR"] + """.concept` c
  JOIN `""" + os.environ["WORKSPACE_CDR"] + """.concept_ancestor` ca
    ON ca.ancestor_concept_id = c.concept_id
  JOIN `""" + os.environ["WORKSPACE_CDR"] + """.observation` o
    ON o.observation_concept_id = ca.descendant_concept_id
  WHERE c.concept_name = 'Consent PII'
    AND c.concept_class_id = 'Module'
  GROUP BY o.person_id
),
breast_cancer_concepts AS (
  -- Cancer du sein (4157332 + descendants via Dataset Builder)
  SELECT DISTINCT c.concept_id
  FROM `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c
  JOIN (
    SELECT CAST(cr.id AS STRING) AS id
    FROM `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr
    WHERE cr.concept_id IN (4157332)
      AND cr.full_text LIKE '%_rank1]%'
  ) a
    ON (c.path LIKE CONCAT('%.', a.id, '.%')
        OR c.path LIKE CONCAT('%.', a.id)
        OR c.path LIKE CONCAT(a.id, '.%')
        OR c.path = a.id)
  WHERE c.is_standard = 1
    AND c.is_selectable = 1
)

SELECT
  person.person_id,
  p_gender_concept.concept_name AS gender,
  person.birth_datetime AS date_of_birth,
  p_ethnicity_concept.concept_name AS ethnicity,
  p_self_reported_category_concept.concept_name AS self_reported_category,
  en.inclusion_date
FROM `""" + os.environ["WORKSPACE_CDR"] + """.person` person
JOIN enrollment en
  ON en.person_id = person.person_id
LEFT JOIN `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept
  ON person.gender_concept_id = p_gender_concept.concept_id
LEFT JOIN `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity_concept
  ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id
LEFT JOIN `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_self_reported_category_concept
  ON person.self_reported_category_concept_id = p_self_reported_category_concept.concept_id
WHERE person.gender_concept_id = 45878463  -- femmes uniquement
  AND NOT EXISTS (
    -- Exclure si cancer du sein AVANT (ou le jour de) l'inclusion
    SELECT 1
    FROM `""" + os.environ["WORKSPACE_CDR"] + """.condition_occurrence` co
    JOIN breast_cancer_concepts bc
      ON bc.concept_id = co.condition_concept_id
    WHERE co.person_id = person.person_id
      AND co.condition_start_date IS NOT NULL
      AND co.condition_start_date < en.inclusion_date
  )
"""

dataset_no_prevalent_breast_cancer_women_df = pandas.read_gbq(
    dataset_no_prevalent_breast_cancer_women_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook"
)

dataset_no_prevalent_breast_cancer_women_df.head(5)


Downloading:   0%|          | 0/380869 [00:00<?, ?rows/s]

,person_id,gender,date_of_birth,ethnicity,self_reported_category,inclusion_date
0,1777333,Female,1932-06-15 00:00:00+00:00,What Race Ethnicity: Race Ethnicity None Of These,None of these,2022-07-19
1,5647222,Female,1934-06-15 00:00:00+00:00,What Race Ethnicity: Race Ethnicity None Of These,None of these,2020-12-11
2,5016061,Female,1935-06-15 00:00:00+00:00,What Race Ethnicity: Race Ethnicity None Of These,None of these,2022-09-07
3,2287629,Female,1942-06-15 00:00:00+00:00,What Race Ethnicity: Race Ethnicity None Of These,None of these,2019-04-18
4,4372683,Female,1943-06-15 00:00:00+00:00,What Race Ethnicity: Race Ethnicity None Of These,None of these,2023-09-12


In [4]:
dataset_breast_cancer_5y_person_df['has_BC'] = 1
dataset_no_prevalent_breast_cancer_women_df['has_BC'] = 0

In [5]:
import pandas as pd

cols_to_keep = ['person_id', 'date_of_birth', 'has_BC', 'inclusion_date']
df1_filtered = dataset_breast_cancer_5y_person_df[cols_to_keep]
df2_filtered = dataset_no_prevalent_breast_cancer_women_df[cols_to_keep]

# 2. Concaténer verticalement
Women_bc_ctr_concat = pd.concat([df1_filtered, df2_filtered], ignore_index=True)

# 1) Forcer les 2 colonnes au même type (datetime pandas)
Women_bc_ctr_concat['date_of_birth'] = pd.to_datetime(
    Women_bc_ctr_concat['date_of_birth'], errors='coerce', utc=True
).dt.tz_convert(None)

Women_bc_ctr_concat['inclusion_date'] = pd.to_datetime(
    Women_bc_ctr_concat['inclusion_date'], errors='coerce'
)

# 2) Calcul âge (années révolues)
Women_bc_ctr_concat['age_at_inclusion'] = (
    (Women_bc_ctr_concat['inclusion_date'] - Women_bc_ctr_concat['date_of_birth'])
    .dt.days // 365
)

Women_bc_ctr_concat


,person_id,date_of_birth,has_BC,inclusion_date,age_at_inclusion
0,3088025,1935-06-15,1,2019-06-12,84
1,1010524,1945-06-15,1,2018-07-27,73
2,4591164,1959-06-15,1,2021-04-28,61
3,1230270,1947-06-15,1,2018-06-11,71
4,8160515,1976-06-15,1,2021-05-12,44
...,...,...,...,...,...
388784,6894329,2004-06-15,0,2023-04-20,18
388785,1589766,2004-06-15,0,2023-07-03,19
388786,7312051,2005-06-15,0,2023-09-05,18
388787,5887313,2005-06-15,0,2023-09-15,18


In [6]:
!gsutil -u $GOOGLE_PROJECT cp gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/ancestry/ancestry_preds.tsv .


Copying gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/ancestry/ancestry_preds.tsv...
| [1 files][163.4 MiB/163.4 MiB]                                                
Operation completed over 1 objects/163.4 MiB.                                    


In [7]:
ancestry_pred = pd.read_csv("ancestry_preds.tsv", delimiter="\t")
ancestry_pred["pca_features"] = ancestry_pred["pca_features"].str[1:-1]
PCs = ancestry_pred["pca_features"].str.split(",", n = 16, expand = True)
PCs = PCs.astype(float)
pid = ancestry_pred[["research_id","ancestry_pred"]]

columns= ["PC1","PC2","PC3","PC4","PC5","PC6","PC7","PC8","PC9","PC10","PC11","PC12","PC13","PC14","PC15","PC16"]
PCs.columns = columns
PCs_final = pd.concat([pid, PCs], axis = 1)

pheno_final = Women_bc_ctr_concat.merge(PCs_final, left_on = "person_id", right_on = "research_id", how = "left").drop(["research_id","date_of_birth"], axis = 1)
pheno_final

,person_id,has_BC,inclusion_date,age_at_inclusion,ancestry_pred,PC1,PC2,PC3,PC4,PC5,...,PC7,PC8,PC9,PC10,PC11,PC12,PC13,PC14,PC15,PC16
0,3088025,1,2019-06-12,84,eur,0.095423,0.129710,0.010196,0.041379,0.004235,...,-0.017359,-0.002958,-0.000500,0.001641,-0.000411,0.000689,0.000134,-0.001173,-0.000986,0.000245
1,1010524,1,2018-07-27,73,eur,0.097735,0.127187,0.012935,0.040561,0.003790,...,-0.013536,-0.002006,0.001373,-0.001060,-0.000413,0.001329,0.001254,-0.000053,0.001078,0.001035
2,4591164,1,2021-04-28,61,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1230270,1,2018-06-11,71,eur,0.098365,0.123402,0.013821,0.033749,0.001227,...,-0.012194,-0.002937,-0.001655,-0.000294,-0.000547,-0.001617,-0.000549,-0.000954,0.001847,0.000471
4,8160515,1,2021-05-12,44,eur,0.097295,0.135378,0.012000,0.054270,-0.001906,...,0.011192,0.001698,0.000029,-0.000856,0.000696,0.000217,-0.002255,0.000125,-0.000623,0.000866
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
388784,6894329,0,2023-04-20,18,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
388785,1589766,0,2023-07-03,19,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
388786,7312051,0,2023-09-05,18,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
388787,5887313,0,2023-09-15,18,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
pheno_final[pheno_final['PC1'].notna()].value_counts(['ancestry_pred','has_BC'])

ancestry_pred  has_BC
eur            0         131853
amr            0          50417
afr            0          47402
eas            0           6090
eur            1           4833
sas            0           2850
afr            1            969
mid            0            790
amr            1            736
eas            1            135
sas            1             33
mid            1             17
Name: count, dtype: int64

In [11]:
import os
import subprocess

# Enregistre le DataFrame `df_with_breast_cancer` dans un fichier TSV local
destination_filename = 'df_bc_5years_cov_389k_nb3.tsv'
pheno_final.to_csv(destination_filename, index=False)

# Récupère le nom du bucket Google Cloud depuis la variable d’environnement
my_bucket = os.getenv('WORKSPACE_BUCKET')

# Copie le fichier TSV local dans le dossier "Data" du bucket
args = ["gsutil", "cp", f"./{destination_filename}", f"{my_bucket}/Data/"]
output = subprocess.run(args, capture_output=True)

# Affiche les éventuelles erreurs retournées par gsutil
output.stderr

b'Copying file://./df_bc_5years_cov_389k_nb3.tsv [Content-Type=text/tab-separated-values]...\n/ [0 files][    0.0 B/ 93.8 MiB]                                                \r-\r- [0 files][  2.6 MiB/ 93.8 MiB]                                                \r\\\r|\r| [0 files][  4.6 MiB/ 93.8 MiB]                                                \r/\r/ [0 files][  6.4 MiB/ 93.8 MiB]                                                \r-\r- [0 files][  8.2 MiB/ 93.8 MiB]                                                \r\\\r|\r| [0 files][ 10.0 MiB/ 93.8 MiB]                                                \r/\r/ [0 files][ 11.9 MiB/ 93.8 MiB]                                                \r-\r\\\r\\ [0 files][ 13.7 MiB/ 93.8 MiB]                                                \r|\r| [0 files][ 15.5 MiB/ 93.8 MiB]                                                \r/\r/ [0 files][ 17.3 MiB/ 93.8 MiB]                                                \r-\r\\\r\\ [0 files][ 19.1 MiB/ 93.8 MiB]      